# Reconciliation Report

Read-only summary comparing model version counts, aliases, and status between source and target catalogs. Nothing is changed or deleted.

## Configuration

Read widget parameters: source/target catalogs, schema, import volume, and volume catalog. Model list is derived from the import manifest; falls back to model_names widget if manifest is missing.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("target_schema", "default", "Target schema")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")
dbutils.widgets.text("volume_catalog", "", "Catalog holding import volume (empty = target_catalog)")
dbutils.widgets.text("model_names", "", "Optional fallback: comma-separated model names if manifest is missing")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()
VOLUME_CATALOG = dbutils.widgets.get("volume_catalog").strip() or TARGET_CATALOG
MODEL_NAMES_FALLBACK = dbutils.widgets.get("model_names").strip()

if not SOURCE_CATALOG or not TARGET_CATALOG:
    raise ValueError("Set source_catalog and target_catalog")

VOLUME_ROOT = f"/Volumes/{VOLUME_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"
print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target: {TARGET_CATALOG}.{TARGET_SCHEMA}")
print(f"Volume: {VOLUME_ROOT}")

## Derive model list

Reads the model list from the import manifest. Falls back to the model_names widget if the manifest is not found.

In [ ]:
import json
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

manifest_path = f"{VOLUME_ROOT}/manifest.json"
try:
    with open(manifest_path) as f:
        manifest = json.load(f)
    MODELS = [m.get("target", "").split(".")[-1] for m in manifest.get("models", []) if m.get("target")]
    MODELS = [n for n in MODELS if n]
    print(f"Models from manifest: {MODELS}")
except FileNotFoundError:
    if MODEL_NAMES_FALLBACK:
        MODELS = [s.strip() for s in MODEL_NAMES_FALLBACK.split(",") if s.strip()]
        print(f"Models from fallback widget: {MODELS}")
    else:
        raise FileNotFoundError(
            f"Manifest not found at {manifest_path} and no model_names fallback provided."
        ) from None

if not MODELS:
    raise ValueError("Model list is empty: check manifest or model_names.")
print(f"Total models to reconcile: {len(MODELS)}")

## Reconciliation: source vs target

For each model, counts versions in both catalogs, lists aliases, and flags any mismatches. Prints a summary table at the end.

In [ ]:
KNOWN_ALIASES = ["Champion", "Challenger", "Shadow"]
results = []

for model_name in MODELS:
    src_full = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{model_name}"
    tgt_full = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{model_name}"

    src_versions = []
    try:
        src_versions = list(client.search_model_versions(f"name='{src_full}'"))
    except Exception as e:
        print(f"  {src_full}: {e}")

    tgt_versions = []
    try:
        tgt_versions = list(client.search_model_versions(f"name='{tgt_full}'"))
    except Exception as e:
        print(f"  {tgt_full}: {e}")

    src_aliases = {}
    tgt_aliases = {}
    for alias in KNOWN_ALIASES:
        try:
            mv = client.get_model_version_by_alias(src_full, alias)
            src_aliases[alias] = mv.version
        except Exception:
            pass
        try:
            mv = client.get_model_version_by_alias(tgt_full, alias)
            tgt_aliases[alias] = mv.version
        except Exception:
            pass

    match = len(src_versions) == len(tgt_versions)
    results.append({
        "model": model_name,
        "src_versions": len(src_versions),
        "tgt_versions": len(tgt_versions),
        "match": match,
        "src_aliases": src_aliases,
        "tgt_aliases": tgt_aliases,
    })

print("\n" + "=" * 80)
print(f"{'Model':<30} {'Source':>8} {'Target':>8} {'Match':>8}  Aliases (src -> tgt)")
print("-" * 80)
for r in results:
    alias_str = "  ".join(
        f"{a}: v{r['src_aliases'].get(a, '-')} -> v{r['tgt_aliases'].get(a, '-')}"
        for a in KNOWN_ALIASES
        if a in r['src_aliases'] or a in r['tgt_aliases']
    ) or "—"
    status = "OK" if r["match"] else "MISMATCH"
    print(f"{r['model']:<30} {r['src_versions']:>8} {r['tgt_versions']:>8} {status:>8}  {alias_str}")

total_src = sum(r["src_versions"] for r in results)
total_tgt = sum(r["tgt_versions"] for r in results)
mismatches = sum(1 for r in results if not r["match"])
print("-" * 80)
print(f"{'TOTAL':<30} {total_src:>8} {total_tgt:>8} {'ALL OK' if mismatches == 0 else f'{mismatches} MISMATCH':>8}")
print("=" * 80)
print("\nReconciliation (read-only) done.")